# Aries's Project
----

## Big Picture Goal:

Of the ~100,000 galaxies in the CEERS field, there are just 2 confirmed at z>10. We suspect there are many more galaxies at z>10 but they are not bright enough for their light to reach our detectors. We measure the UV Luminosity, or the intrinsic brightness, of the z>10 CEERS sample and the remaining CEERS sample to learn how intrinsicaly bright these z>10 galaxies are compared to our foreground galaxy population.


## Project outline:



- **Step 1: Reading in the Photometric Data** We load in out photometry catalog and get comfortable with indexing the data through conditional arrays
  
- **Step 2:Identifying the z>10 Galaxies** Read in the data for the galaxies in CEERS at z>10.
- **Step 3:Matching to Photometry** We find the photometric sources that match the high-z CEERS galaxies
- **Step 4: Measuring the UV Continuum Luminosity** We measure the UV continuum at 2800 Angstroms for the z>10 and z<10 galaxy samples
- **Step 5: Make your Plot**: We plot the galaxy luminosities and redshifts to learn if the z>10 galaxies are more luminous than the remaining sample.


Go through the following notebook step by step. Feel free to get stuck and ask me questions! This represents my intelectual work. I put effort in to making these projects fun and fine-tuned to your interests. Please do not upload my ideas or code in to any large language model. I encourage you to ask chat bots questions when you find it helps, but please keep my personal code and ideas seperarted from what you ask your prefered model. 

In [1]:
# Importing the basics we need

import astropy.units as u
import astropy.constants as c
from astropy.coordinates import SkyCoord, search_around_sky
from astropy.time import Time
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd
import glob
#un-comment this to change the matplotlib interface to be a little more interactive if that is your preference
#%matplotlib notebook
%matplotlib inline
from astropy.io import fits
from itertools import combinations
import pickle
from astropy.cosmology import WMAP9 as cosmo
import seaborn as sns
from astropy.table import Table


#Setting up matplotlib how I like it - you can change this to reflect your preferences.
plt.rcParams['figure.figsize'] = (10, 10)
plt.rc('axes', labelsize=27)
#plt.rc('axes', labelweight='bold')
plt.rc('axes', titlesize=27)
plt.rc('axes', titleweight='bold')
plt.rc('font', family='sans-serif')
plt.rcParams.update({'font.size': 20})

# Step 1: Reading the Photometric Data

When I send you files, they will always be easily read in as a pandas DataFrame. You'll learn as you go through physics and astronomy that every one has a different opinion on the best file format to exchange science data with. Everyone uses a different file type, method for reading files in, and data format. Everyone also belevies that thier method is the correct method and all other methods are inferior. I am no exception and refuse to change my file writing/reading methods.  

Pandas DataFrames are pretty standard in data science and learning to work with them will make your life a lot easier. Let's read in the files we need and run through an example.


In [ ]:
#File paths- CHANGE TO YOURS


#make sure the absolute path to your data preceds the file string.
all_ceers = pd.read_pickle('df_new.pkl')


Let's start with just printing out the Data Frame:

In [ ]:
all_ceers

Big Data! Let's take a look at the keyword in the Data Frame. 

In [ ]:
all_ceers.keys()

Lots of keys here. We will concentrate on RA (right ascension in decimal degrees), DEC (declination in decimal degrees), and PHOTOM_RED_SHIFT (the photometric red shift or line of sight distance) Let's build some arrays that carry just these around.

In [ ]:
ra_ceers = all_ceers['RA'].values
dec_ceers = all_ceers['DEC'].values
z_ceers = all_ceers['PHOTOM_RED_SHIFT'].values

You'll also be working with matplotlib to generate graphs. It has a lot of great functionality in Python. We can make a simple plot of CEERS using matplotlib's scatter plot:

In [ ]:
#Calling the scatter plot function
plt.scatter(ra_ceers,dec_ceers, s = .1)


#Adding x labels, y labels, and a title
plt.xlabel('RA [deg]')
plt.ylabel('DEC [deg]')
#By standard convention we always dsiplay RA backwards!
plt.gca().invert_xaxis()
plt.title('CEERS'); #End yout matplotlib statements with a semicolon to prevent it from printing every output

Another helpful tool for you will be boolean arrays. You can use thes to index larger arrays and slice them in to smaller ones. Let's build an array that looks at redshifts >5:

In [ ]:
bool1 = all_ceers['PHOTOM_RED_SHIFT'] >5

And index our values with the boolean:

In [ ]:
ra_g4 = ra_ceers[bool1]
dec_g4 = dec_ceers[bool1]

Let's plot them over our last plot, now in another color, so we can see where they are:

In [ ]:
#Calling the scatter plot function
plt.scatter(ra_ceers,dec_ceers, s = .1, label = 'All Galaxies')

#Adding our new data and giving it a label for the legend
plt.scatter(ra_g4, dec_g4, c = 'r', s = .1, label = 'z>4')


#Adding x labels, y labels, and a title
plt.xlabel('RA [deg]')
plt.ylabel('DEC [deg]')
#Call the legend to display it
plt.legend()
#By standard convention we always dsiplay RA backwards!
plt.gca().invert_xaxis()
plt.title('CEERS'); #End yout matplotlib statements with a semicolon to prevent it from printing every output

Wow lots of sources! But let's refine our bins and look at sources that are also at redshifts <5. We will add another array:

In [ ]:
bool2 = all_ceers['PHOTOM_RED_SHIFT'] <5

And combine them with '&', which will grab the values that are true in both booleans. If we wanted all values that were true in both arrays instead, we would combine them with '|'. 

In [ ]:
ra_g4l5 = ra_ceers[bool1 & bool2] 
dec_g4l5 = dec_ceers[bool1 & bool2]

In [ ]:
#Calling the scatter plot function
plt.scatter(ra_ceers,dec_ceers, s = .1)

#Adding our new data and giving it a label for the legend
plt.scatter(ra_g4l5, dec_g4l5, c = 'r', s = .3, label = 'z>4')


#Adding x labels, y labels, and a title
plt.xlabel('RA [deg]')
plt.ylabel('DEC [deg]')
#Call the legend to display it
plt.legend()
#By standard convention we always dsiplay RA backwards!
plt.gca().invert_xaxis()
plt.title('CEERS'); #End yout matplotlib statements with a semicolon to prevent it from printing every output

Very cool! There's been a lot of suspicion about the super high redshift sources at z>10. Remake this plot for sources of redshifts 10-12. Are they in any particular area of CEERS? Do you notice any differences?

In [ ]:
#---YOUR WORK---

#Feel free to copy/paste some stuff from above and make your own plot!

# Step 2: Identifying the Spectroscopically Confirmed z>10 Galaxies

The CEERS high-z sample:

- https://iopscience.iop.org/article/10.3847/2041-8213/acdd54

We will use this paper to grab the z>10 spectroscopically confirmed galaxies.

There are more in other fields but, for now, we concentrate on CEERS. Read in the galaxy data and plot the spectra.

In [8]:
#Read in this file for info about the NIRSpec confirmed CEERS galaxies

nirspec_dat = pd.read_csv('YOUR PATH HERE /CEERS_NIRSpec_MSA_catalog_dr0.9_allinfo_v3.csv', delimiter=';')

We can print out the keys in the data frame to learn what information we have:

In [14]:
nirspec_dat.keys()

Index(['MSA_ID', 'ID', 'ra', 'dec', 'z_phot', 'F606W', 'F814W', 'F125W',
       'F160W', 'F115W', 'F150W', 'F200W', 'F277W', 'F356W', 'F410M', 'F444W',
       'prism_4', 'Mgrat_4', 'prism_5', 'Mgrat_5', 'prism_7', 'Mgrat_7',
       'prism_8', 'Mgrat_8', 'prism_9', 'Mgrat_9', 'prism_10', 'Mgrat_10',
       'prism_11', 'prism_12', 'prism_DD', 'prism_z', 'prism_zq', 'Mgrat_z',
       'Mgrat_zq', 'allvet_prism_z', 'allvet_prism_zq', 'allvet_Mgrat_z',
       'allvet_Mgrat_zq', 'allvet_prism_features', 'allvet_prism_comments',
       'allvet_prism_reviewer', 'allvet_Mgrat_features',
       'allvet_Mgrat_comments', 'allvet_Mgrat_reviewer'],
      dtype='object')

In [17]:
#The galaxy IDs

nirspec_dat['MSA_ID']

0            0
1        10010
2         1019
3         1023
4         1025
         ...  
1481     D9597
1482     D9600
1483    D98870
1484     D9932
1485     D9973
Name: MSA_ID, Length: 1486, dtype: object

In [ ]:
# The Redshifts

nirspec_dat['prism_z']

In [ ]:
# Create a boolean aray to index the nirspec data frame and only include sources with z>10

sources_zg10 = ?????

In [18]:
#Index this same array to exclude sources 13452 and 87379. These are low-confidence detections.

sources_zg10 = ?????

In [ ]:
# For each of your remaining source, print out the sources ID, spectroscopic and photometric redshifts (keyword 
# is z_phot), and the RA and DEC coordinates.

#Loop through all the sources
for m in sources_zg10['MSA_ID']:
    #This variable stores the MPTID
    MPTID = m
    #This variable contains all the information in the bigger data frame, but for just the
    #source we are iterating through on this loop.
    source_data = sources_zg10[sources_zg10['MSA_ID'] == m]
    #Put some code here to make your print statements. Reference the keys above or make some
    #print statements to figure out what the data looks like to figure out the apropriate structure here.
    print(??????)
    

In [ ]:
#Turn your printed statements in to arrays! (Not justt the RA/dec values, you'll want everything)

#copy/paste your code from the previous cell down here and adjust the code so what was a print statement is now being appended to a list. 
#That structure will look like:

ra_list = []
dec_list = []
.....


for.......

    ra_value = #the thing you were previously printing
    dec_value = #the thing you were previously printing

    ra_list.append(ra_value)
    dec_list.appemnd(dec_value

#Convert your lists to arrays
ra_array = np.array(ra_list)
dec_array = np.array(dec_list)
....

Before moving to the next step, you should have arrays with source IDs, RA/DEC coordinates, and spectroscopic/photometric redshifts. Print these arrays below to double check:

In [ ]:
#print arrays here by typing the name of the variable they are stored under:



BONUS IF YOU WANT: 

- re-make your CEERS plots from step 1, this time with your confirmed z>10 targets plotted as large red stars.

# Step 3: Matching to Photometry

You've already laoded the photometry in and saved it under the name all_ceers.

In [ ]:
# Print the keys to all_ceers so you can become farmiliar with what is stored where (just like we printed the keys in the pervious step)





Matching the catalogs:

We will use astropy's catlog matching funciton (read more about it here: https://docs.astropy.org/en/stable/coordinates/matchsep.html)

We want to find the closest match in the photometry to our spectroscopic sources

In [ ]:



smallRA = 
smalllDEC =
smallMPT = 
#..... Add other arrays you'd like to match


bigRA = 
bigDEc = 
bigIDS = #you'll want photometry IDs (Object keyword in the big data frame)

#Matching the photometry targets to the closest spectroscopic targets in RA/DEC

#The smaller dataset
c = SkyCoord(ra=np.array(smallRA)*u.degree, dec=np.array(smallDEC)*u.degree)
#The larger dataset we want to match the smaller dataset to 
catalog = SkyCoord(ra=bigRA*u.degree, dec=bigDEC*u.degree)
#The arrays we can use to cross match
idx, d2d, d3d = c.match_to_catalog_sky(catalog)



In [20]:
# Print out some of these arrays. Can you describe breifly what idx and d2d are? We are ignoring d3d




In [ ]:
# Build the big catalog
#Use the idx (index) arrays to re-order your arrays to match up with the bigger data frame

#NOTE: if you're reading data in from a pandas dataframe that came from a FITS file, you may have byte order issues when
#you try to combine it with anoteher type. If you run in to this order, simply reorder the bytes by passing:
#.byteswap().newbyteorder()

match_catalog = pd.DataFrame({'MPT ID':smallMPT,
                       'RA':smallRA,
                       'DEC': smallDEC, 
              'd2d': d2d, 'Photometry RA': bigRA[idx], 
              'Photometry DEC':bigDEC[idx], 'Photometry ID': bigIDs[idx]})

#Did you remember to add all the keywords you bulilt to the dataframe above? Make sure you aren't missing anything.
#Reset the index so everything is nice and ordered
match_catalog = match_catalog.reset_index()
match_catalog




In [ ]:
# Create a version of your dataframe that only includes the minimum d2c value for each of your high-z sources. 





#Print out the photometric IDs that match your high-z galaxies




Plot the photometry from your galaxy!

Below is a function I wrote to plot the spectral energy distribution, or photometry, for your galaxy. Run the cell so you can
call the function.

In [ ]:
def get_sed(obj, df, log = False, lt = 0, rt = 0):



    import astropy.units as u
    conversion = u.AA.to(u.um)
    conversion

    #Building a dictionary of the filter widths and depths
    width_dict = {60.6: np.array([(5889.2-(2189.2/2))*conversion, (5889.2+(2189.2/2))*conversion])*100, 
                  81.4: np.array([(8039.1-(1565.2/2))*conversion, (8039.1+(1565.2/2))*conversion])*100,
                  105.0: np.array([(1055.2-(265.0/2))*0.001, (1055.2+(265.0/2))*0.001])*100,
                  115.0: np.array([(1.154-(0.225/2))*100, (1.154+(0.225/2))])*100,
                  125.0: np.array([(1248.6-(284.5/2))*0.001, (1248.6+(284.5/2))*0.001])*100,
                  140.0: np.array([(1392.3-(384.0/2))*0.001, (1392.3+(384.0/2))*0.001])*100,
                  150.0: np.array([(1.501-(0.318/2)), (1.501+(0.318/2))])*100,
                  160.0: np.array([(1536.9-(268.3/2))*0.001, (1536.9+(268.3/2))*0.001])*100,
                  200.0: np.array([(1.990-(0.461/2), (1.990+(0.461/2)))])*100,
                  277.0: np.array([(2.786-(0.672/2)), (2.786+(0.672/2))])*100,
                  356.0: np.array([(3.365-(0.347/2)), (3.365+(0.347/2))])*100,
                  410.0: np.array([(4.092-(0.436/2)), (4.092+(0.436/2))])*100,
                  444.0: np.array([(4.421-(1.024/2)), (4.421+(1.024/2))])*100}

    error_dict = {60.6: 2189.2*conversion*100, 
                  81.4: 1565.2*conversion*100,
                  105.0: 265.0*0.001*100,
                  115.0: 0.225*100,
                  125.0: 284.5*0.001*100,
                  140.0: 384.0*0.001*100,
                  150.0: 0.318*100,
                  160.0: 268.3*0.001*100,
                  200.0: 0.461*100,
                  277.0: 0.672*100,
                  356.0: 0.787*100,
                  410.0: 0.436*100,
                  444.0: 1.024*100}
    
    xpos = -40
    ypos = 1000
    let = 'C'
    plt. style. use('default')
    plt.rcParams['figure.figsize'] = (10, 10)
    plt.rc('axes', labelsize=27)
    #plt.rc('axes', labelweight='bold')
    plt.rc('axes', titlesize=30)
    plt.rc('axes', titleweight='normal')
    plt.rc('font', family='sans-serif')
    plt.rcParams.update({'font.size': 20})

    
    plt.rcParams['figure.figsize'] = (10, 5)
    
    
    shape_key = 'Pure Shape'
    dfc = df
    
    

    plt.rcParams['figure.figsize'] = (10, 5)
    #plot_sources(remaining['Object'], remaining, df_col6, 6)
    #df = dfc
    dat = df[df['Object'] == obj]
    #plt.rcParams['figure.figsize'] = (20, 10)
    #Grab filters from DataFrame
    HST_filters = [140.0, 81.4, 60.6, 105.0, 125.0,  160.0]

    ra = []
    dec = []
    zs = []
    ps = []
    #redshifts = []
    fs = []
    fs_strings = []
    a=dfc.index[0]
    for key in df.loc[a][3:16].keys():
        fs.append(float(key.split('F')[1]))
        fs_strings.append(key)
    start=3
    stop=16
    stop2 = 29
    f_final = []
    for f in fs:
        if f>600:
            f_final.append(f*.1)
        else:
            f_final.append(f)

    #Create a colormap 
    #colormap = plt.cm.gist_ncar
    #colors = [colormap(i) for i in np.linspace(0, 1,len(sources)+2)][1:-1]
    #Create an SED for each of the sources
    i=0
    n=0
    rss = []
   # for obj in sources[0:45]:

   # props = dict(boxstyle = 'square', facecolor = 'wheat', alpha = 0.2)
   ##### plt.text(xpos, ypos, let, size = 'large', bbox = props)

    #Increase the iteration tracker, grab the DataFrame for just this object
    n+=1
    #print(shapes[i])
    mk = np.array(df['Object']) == np.array(obj)
    a = df[mk]
    #Get the redshift and filters
    rs = a['PHOTOM_RED_SHIFT'].values[0]
    ra.append(a['RA'].values[0])
    dec.append(a['DEC'].values[0])
    rss.append(rs)
    zs.append(rs)
    filters = np.array(a.keys()[start:stop].values)
    filter_er = np.array(a.keys()[stop:stop2].values)
    #Mask the data so all filter values are greater than 3 times their error
    mask = (a[filters].values[0] > (3*a[filter_er].values[0]))& (a[filter_er].values[0]>0) 
    #Sort the data so the SED plot connects from left to right, regardless of filter input order
    z = sorted(zip(np.array(f_final)[mask],a[filters].values[0][mask]))
    x=[i[0] for i in z]
    y=[i[1] for i in z]

   # print(rs)

    xerror = []
    for j in x:
        xerror.append(error_dict[j]/2)




    plt.errorbar(x,y,a[filter_er].values[0][mask], xerr = xerror, ls='none',
                color = 'k', marker = '*', markersize = 20, zorder = 1, label = 'JWST Data',
                markerfacecolor='r')
        #Add the x ticks
    plt.xticks(np.array(f_final)[mask])

    itr = 0
    xvals= []
    for point in range(len(x)):
        if itr == 0:
            if x[point] in HST_filters:
                itr+=1
                plt.plot(x[point], y[point], marker = '*', color = 'grey', markersize = 20, label = 'HST Data',
                        zorder = 2)
                xvals.append(x[point])
        else:
            if x[point] in HST_filters:
                itr+=1
                plt.plot(x[point], y[point], marker = '*', color = 'grey', markersize = 20)
                xvals.append(x[point])
    plt.axvspan(241.6, 312.7,  color = 'g', alpha = .25)
    plt.axvspan(314, 398,  color = 'y', alpha = .25)
    plt.axvspan(386.4, 430.1,  alpha = 0.25, color = 'orange')
    plt.axvspan(388, 498.6,  color = 'r', alpha = .25)
    plt.axvspan(175.5, 222.6,  color = 'grey', alpha = 0.55)#color = 'cyan', alpha = .25)
    plt.axvspan(133.1, 166.8,  color = 'grey', alpha = 0.45)#color = 'blue', alpha = .25)
    plt.axvspan(101.3, 128.2,  color = 'grey', alpha = 0.25)#color = 'purple', alpha = .25)
    #Add titles and labels
    #plt.title(f'Sources of Interest All pointings Shape {cs[num]}')
    #plt.title(f'Sources of Similar SED Shape \n{n} sources')
    #plt.title(f'z = {np.mean(rss)}')
    plt.ylabel('Flux (nJy)')

    140.0, 81.4, 60.6, 105.0, 125.0,  160.0

    labs = {'81.4':.80, '140.0':1.39, '60.6':.59, '105.0':1.05, '125.0':1.25, '160.0':1.54}
    ticks = [115.4, 150.1, 199.0, 278.6, 356.3, 409.2, 442.1]
    labels = [1.15, 1.50, 1.99, 2.78, 3.56, 4.09, 4.42]
    for v in xvals:
        ticks.append(v)
        labels.append(labs[str(v)])

    plt.xticks(ticks, labels, rotation = 90, fontsize = 14)


   # xlabels = np.around(np.array(x)/100,2)
    if log == True:
        plt.yscale("log")
    plt.xlabel(r'Wavelength ($\mu$m)')
   # plt.xticks(x, xlabels, 
   #            rotation = 90, fontsize = 16)
    r = a['RA'].values[0]
    d = a['DEC'].values[0]
    p = a['Pointing'].values[0]
    ps.append(p)
    #print(r,d)
    tenxs = a['Lowest_Chi2_Z'].values[0]
    tenrs = a['Lowest_Chi2'].values[0]
    ###LEGEND CONTROL
    plt.legend(loc = 'upper left')
    #plt.title(f'{obj}, Redshift: {round(rs,3)} \n Ra: {r} Dec: {d} \n Four most likely z: {tenxs[0:6]}\n Corresponding chisq: {tenrs[0:6]}')
    plt.tick_params('y', length=20, width=2, which='major')
    plt.tick_params('y', length=10, width=1, which='minor')
       # plt.legend()
    plt.title(f'{obj} Zphot = {round(rs,2)}')
    #plt.savefig(f'/home/kelcey/paper_plots/{obj}.pdf', bbox_inches = 'tight')
    i+=1




In [ ]:
#Pass your photometry IDs here to plot the galaxies!
#Do this for all your galaxies so you can verify your identification.

get_sed('Photometry ID goes here', all_ceers)

# Step 4: Measuring the UV Continuum Luminosity



The luminosity is the intrisic brightness of your galaxy! You need to look up a formula from your notes or text book for the luminosity of a galaxy. It should depend on

$D_L$: the luminosity distance 

F: the flux from a galaxy

and some other constants 

You'll also need to convert this by mutiplying by:

$\frac{c}{\lambda(1+z)}$

This takes our measurement out of the observed frame and into the rest frame (dividing by 1+z) and takes us from flux units coresponding to frequency and put us in flux units coresponding to wavelength  (multiplying by c/$\lambda$).



Type your formula here: (double click on this cell to edit the text):

$L = ???$

## Getting the correct photometric filter

We need to identify the correct photometric filter for the continuum flux. We can use 2800 angstroms. We will use astrpy units (imported as u) to handle all of our units for us! The continuum wavelength is (in our micrometer photoemtry units):

In [23]:
continuum_wav = (2800 * u.AA).to(u.um)
continuum_wav

<Quantity 0.28 um>

Next, we need to calculate the observed from the telescope (so the redshifted wavelength). Play around with a few redshift values so you get the idea.

In [ ]:
# Put a target redhsift here from one of your sources:

ztarg = ???

observed_wave = continuum_wav * (1+ztarg)
observed_wave

Finally, we need to identify the photoemtric filter that capture the UV continuum. You can use this dictionary which tells you the wavelength ranges covered by each filter:

In [32]:
#Run this cell to initialize dictionary
width_dict = {
              105.0: np.array([(1055.2-(265.0/2))*0.001, (1055.2+(265.0/2))*0.001]),
              115.0: np.array([(1.154-(0.225/2))*100, (1.154+(0.225/2))]),
              125.0: np.array([(1248.6-(284.5/2))*0.001, (1248.6+(284.5/2))*0.001]),
              140.0: np.array([(1392.3-(384.0/2))*0.001, (1392.3+(384.0/2))*0.001]),
              150.0: np.array([(1.501-(0.318/2)), (1.501+(0.318/2))]),
              160.0: np.array([(1536.9-(268.3/2))*0.001, (1536.9+(268.3/2))*0.001]),
              200.0: np.array([(1.990-(0.461/2), (1.990+(0.461/2)))]),
              277.0: np.array([(2.786-(0.672/2)), (2.786+(0.672/2))]),
              356.0: np.array([(3.365-(0.347/2)), (3.365+(0.347/2))]),
              410.0: np.array([(4.092-(0.436/2)), (4.092+(0.436/2))]),
              444.0: np.array([(4.421-(1.024/2)), (4.421+(1.024/2))])}


width_dict

{105.0: array([0.9227, 1.1877]),
 115.0: array([104.15  ,   1.2665]),
 125.0: array([1.10635, 1.39085]),
 140.0: array([1.2003, 1.5843]),
 150.0: array([1.342, 1.66 ]),
 160.0: array([1.40275, 1.67105]),
 200.0: array([[1.7595, 2.2205]]),
 277.0: array([2.45 , 3.122]),
 356.0: array([3.1915, 3.5385]),
 410.0: array([3.874, 4.31 ]),
 444.0: array([3.909, 4.933])}

In [31]:
#Build some code here to calculate the correct photometry filter that contains the 2800AA continuum

#Initialize a list to store your filters

#For loop that iterates through your high-z sources

    #Get the observed wavelength like you did above:
    ztarg = ??? #hint: this should change as your loop iterates

    observed_wave = continuum_wav * (1+ztarg)


    #Loop through all the filers in the dictionary above
    # for key in width_dict.keys()
        #check if observed_wave is less than the upper bound AND greater than the lower bound
            #If condition is satisfied, throw the key in the list you initialized.

{105.0: array([0.9227, 1.1877]),
 115.0: array([104.15  ,   1.2665]),
 125.0: array([1.10635, 1.39085]),
 140.0: array([1.2003, 1.5843]),
 150.0: array([1.342, 1.66 ]),
 160.0: array([1.40275, 1.67105]),
 200.0: array([[1.7595, 2.2205]]),
 277.0: array([2.45 , 3.122]),
 356.0: array([3.1915, 3.5385]),
 410.0: array([3.874, 4.31 ]),
 444.0: array([3.909, 4.933])}

## Retrieve the flux for your target filter

In [ ]:
#Build some code here to retreive the flux, and error on flux valeus,for the UV continuum of your galaxies.

#Lists to store flux values
contvs = []
contvs_err = []

#for loop that iterates over the photometry IDS of the high-z galaxies
    #tHIS will give you the photometry information for your target galaxy only
    galaxydat = all_eelgs[all_eelgs['Object'] == YOUR PHOTOMETRY ID FROM THE LOOP GOES HERE]
    #Create a variable for the keyword you need
    #Index galaxydat to grab just the apropriate photometric filter
    #append the flux and error to the lists


In [ ]:
Lcont = []#a list that will hold our continuum luminosities
Lcont_e = [] # a list that will hold the error on these luminosities

rs =  #your list of redshift values

for i in range(len(rs)):
    con = contvs[i]*u.nJy#adding our correct flux units
    cont_errcontvs_err[i]*u.nJy
    #We getthe luminosity distance from the redshift with astropy! Nice and easy.
    d = cosmo.luminosity_distance(rs[i])
    #Use your luminosity formula and fluxes we calculated to calculate the luminosity!
    Lc = ?????? * (c.c) * (1/( (1+rs[i]) * (continuum_wav))) 
    Le = ????? * (c.c) * (1/((1+rs[i]) * continuum_wav))
    #We decompose the astropy units quantity into base CGS units (astronomy standard)
    Lcont.append(Lc.decompose(bases=u.cgs.bases).value)#units should be u.erg/u.s/u.Hz
    #Do the same for the errro
    Lcont_eappend(???????)

Repeat these calculations for your background population all_ceers and build arrays for all CEERS galaxies

In [ ]:
# Write your code here! You'll combine all the things we did in the previous steps but now with all the CEERS galaxies

#If you don't know where to start, try writing pseudo code like what I gave you in each step! 

#Your end goal is Lcont, Lcont_e and redshift arrays for all the galxies in all_ceers.

# Step 5: Making the plot
use the arrays you created in the previous step to plot the UV luminositeis and redshifts!


- x axis: redshift (z)
- y axis: UV luminosity

color code the plot and include error bars on your UV luminosities

In [ ]:




#Plot the lower-z sources

plt.scatter(#x data,
    #y data,
    #color = ,
    #label = ,
    #
            )

#Plot the z>10 sources

plt.scatter(#x data,
    #y data,
    #color = ,
    #label = ,
    #
            )


#call plt.errorbar() to add error bars to plot

plt.title('Title goes here')

plt.legend()



BONUS IF TIME ALLOWS:

(short)

use astropy to calulate the age of the universe when your galaxy emitted it's light. Create anotehr version of your plot
with this as the x axis.


(longer)

- I will give you the NIRSPec data for the z>10 galaxies
- You will plot these galaxies and measure the slope of their UV continuum!
- You will compare this to the photometric population